# XAI Pipeline — PlantVillage leaf-disease CNN

RGB leaf images, 15 disease classes (softmax).

All logic lives in `model/` and `xai/`. Train the model first with `python train_leaf.py` from the repository root, then run this notebook top to bottom.

## 1 · Setup

In [ ]:
import os, sys, json
from collections import defaultdict
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from model.runtime import setup
from xai.common import replace2linear
from xai.runner import ALL_METRICS, run_evaluation
from xai.visualization import plot_training_history, plot_cam_layers, plot_attribution_maps

from tf_keras_vis.utils.scores import CategoricalScore
from model.loaders.leaf_disease import (create_data_generators, create_dataframe_from_directory,
                                        load_background, preprocess_input)
from xai import leaf

setup(42)

In [ ]:
DATA_DIR     = 'data/PlantVillage'
MODEL_PATH   = 'models/leaf.keras'
HISTORY_PATH = 'models/leaf_history.json'
CSV_OUT      = 'results/leaf_metrics.csv'
IMAGE_SIZE   = 64

## 2 · Data

In [ ]:
train_df, val_df, test_df = create_dataframe_from_directory(DATA_DIR)
train_gen, val_gen, test_gen = create_data_generators(DATA_DIR, train_df, val_df, test_df, IMAGE_SIZE)
class_names = sorted(train_gen.class_indices, key=train_gen.class_indices.get)
print(f'Train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)} | {len(class_names)} classes')

## 3 · Model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
with open(HISTORY_PATH) as f:
    plot_training_history(json.load(f))

## 4 · Classification performance

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, steps=len(test_gen))
y_pred = np.argmax(model.predict(test_gen, steps=len(test_gen), verbose=0), axis=1)
y_true = test_gen.classes

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix')
plt.tight_layout(); plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

## 5 · XAI setup

In [ ]:
background = load_background(DATA_DIR, train_df, 100, IMAGE_SIZE)
evaluator = leaf.XAIEvaluator(model, img_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), fractions=10, robustness_n=10,
                              sensitivity_iters=5, selectivity_patches=10)
leaf.register_all(evaluator, background)
print('Explainers:', list(evaluator.explainers))

## 6 · Grad-CAM and Grad-CAM++ across layers

In [ ]:
img_array = tf.cast(preprocess_input(os.path.join(DATA_DIR, test_df.iloc[0]['File']), (IMAGE_SIZE, IMAGE_SIZE)), tf.float32)
class_index = int(np.argmax(model.predict(img_array, verbose=0)))
print('Predicted:', class_names[class_index])
to_display = lambda img: np.clip(np.asarray(img), 0, 1)
plot_cam_layers(model, img_array, CategoricalScore(class_index), leaf.LAYER_INDICES,
                to_display(img_array[0]), replace2linear)

## 7 · Attribution maps for confident predictions

In [ ]:
VIZ_CLASSES = class_names[:1]

CONF_THRESH = 0.9
candidates = defaultdict(list)
for batch_imgs, _ in (val_gen[i] for i in range(len(val_gen))):
    batch_imgs = np.asarray(batch_imgs)
    preds = model.predict(batch_imgs, verbose=0)
    pred_idxs, confs = np.argmax(preds, axis=1), np.max(preds, axis=1)
    for img, pred_idx, conf in zip(batch_imgs, pred_idxs, confs):
        cls_name = class_names[pred_idx]
        if cls_name in VIZ_CLASSES and not candidates[cls_name] and CONF_THRESH <= conf <= CONF_THRESH + 0.11:
            candidates[cls_name].append((img, conf))
    if all(candidates[c] for c in VIZ_CLASSES):
        break
print({c: [round(float(conf), 3) for _, conf in v] for c, v in candidates.items()})

In [ ]:
for cls_name, items in candidates.items():
    for img, conf in items:
        plot_attribution_maps(evaluator, img, class_names.index(cls_name), to_display(img),
                              list(evaluator.explainers), f'{cls_name} (confidence {conf:.2f})')

## 8 · Quantitative evaluation

In [ ]:
images = leaf.get_balanced_sample(val_gen, num_samples=2)
results = run_evaluation(evaluator, images, ALL_METRICS, None, CSV_OUT)
results